# Sistemas de equações lineares

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/04-sistemas-lineares.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. explicar a eliminação de Gauss como uma fatoração $PA = LU$ e estimar o seu custo;
> 2. explicar por que o pivotamento é obrigatório;
> 3. usar o **número de condição** para prever quantos algarismos uma solução pode perder, e distinguir resíduo pequeno de erro pequeno;
> 4. escolher entre um método direto, um método iterativo clássico e o gradiente conjugado para um sistema grande e esparso;
> 5. escrever um critério de parada que garante a precisão pedida.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
from scipy import sparse
from scipy.sparse.linalg import spsolve
from cnlivro.lineares import (eliminacao_gauss, lu, resolve_lu, gauss_seidel, jacobi,
                              gradiente_conjugado, placa_laplace)

## 1. Problema de engenharia: a temperatura numa placa

Uma placa metálica quadrada, de lado 1 m, faz parte de um trocador de calor. A borda de cima fica em contato com um fluido a $100\ ^\circ$C, e as outras três bordas são mantidas a $0\ ^\circ$C. Em regime permanente, qual é a temperatura em cada ponto da placa? Um sensor será instalado no centro, e o projeto exige conhecer essa temperatura com precisão de $0{,}001\ ^\circ$C.

A temperatura $T(x, y)$ obedece à **equação de Laplace**, $\partial^2 T/\partial x^2 + \partial^2 T/\partial y^2 = 0$. O capítulo 15 trata dessa equação em detalhe. Aqui basta uma ideia: cobrimos a placa com uma grade de $N \times N$ pontos internos, e em cada ponto a temperatura é a **média dos quatro vizinhos**:

$$
T_{i,j} = \frac{T_{i-1,j} + T_{i+1,j} + T_{i,j-1} + T_{i,j+1}}{4}.
$$ (eq-placa)

São $N^2$ equações lineares com $N^2$ incógnitas. Com $N = 100$, são **10 000** equações. Resolver sistemas assim, rapidamente e com precisão garantida, é o assunto deste capítulo.

## 2. A ideia em uma imagem

Cada ponto da grade gera **uma linha** da matriz $A$. Como cada ponto só conversa com os quatro vizinhos, cada linha tem no máximo cinco números diferentes de zero. A matriz é enorme, mas quase vazia: é uma matriz **esparsa**.

In [ ]:
N = 4
fig = make_subplots(rows=1, cols=2, column_widths=[0.42, 0.58], horizontal_spacing=0.12,
                    subplot_titles=("a grade da placa (N = 4)", "a matriz A (16 × 16)"))
ii, jj = np.meshgrid(np.arange(N), np.arange(N), indexing="ij")
fig.add_trace(go.Scatter(x=jj.ravel(), y=ii.ravel(), mode="markers+text", text=[str(k) for k in range(N * N)],
                         textposition="top right", marker=dict(size=12, color=CORES["referencia"]),
                         showlegend=False, hoverinfo="skip"), 1, 1)
for x0, y0, x1, y1 in [(-1, -1, N, -1), (-1, N, N, N), (-1, -1, -1, N), (N, -1, N, N)]:
    fig.add_trace(go.Scatter(x=[x0, x1], y=[y0, y1], mode="lines", showlegend=False, hoverinfo="skip",
                             line=dict(color=CORES["erro"] if y0 == N and y1 == N else CORES["aproximacao"], width=4)), 1, 1)
centro, viz = (1, 2), [(0, 2), (2, 2), (1, 1), (1, 3)]
fig.add_trace(go.Scatter(x=[centro[1]], y=[centro[0]], mode="markers", marker=dict(size=20, color=CORES["destaque"]),
                         showlegend=False, hoverinfo="skip"), 1, 1)
fig.add_trace(go.Scatter(x=[v[1] for v in viz], y=[v[0] for v in viz], mode="markers",
                         marker=dict(size=16, color=CORES["secundaria"]), showlegend=False, hoverinfo="skip"), 1, 1)
fig.add_annotation(x=1.5, y=N + 0.45, text="borda a 100 °C", showarrow=False, font=dict(color=CORES["erro"]), row=1, col=1)
fig.add_annotation(x=1.5, y=-1.45, text="bordas a 0 °C", showarrow=False, font=dict(color=CORES["aproximacao"]), row=1, col=1)
A4, _ = placa_laplace(N)
A4 = A4.toarray()
linha = centro[0] * N + centro[1]
lin, col = np.nonzero(A4)
cores = [CORES["destaque"] if (l == linha and c == linha) else CORES["secundaria"] if l == linha
         else CORES["exata"] for l, c in zip(lin, col)]
fig.add_trace(go.Scatter(x=col, y=lin, mode="markers", marker=dict(size=9, color=cores, symbol="square"),
                         showlegend=False, hovertext=[f"A[{l},{c}] = {numero(A4[l, c])}" for l, c in zip(lin, col)],
                         hoverinfo="text"), 1, 2)
fig.add_hrect(y0=linha - 0.5, y1=linha + 0.5, line_width=0, fillcolor=CORES["destaque"], opacity=0.15, row=1, col=2)
fig.update_xaxes(visible=False, range=[-1.6, N + 0.6], row=1, col=1)
fig.update_yaxes(visible=False, range=[-1.8, N + 0.8], scaleanchor="x", row=1, col=1)
fig.update_xaxes(range=[-0.5, 15.5], dtick=3, title="coluna", row=1, col=2)
fig.update_yaxes(range=[15.5, -0.5], dtick=3, title="linha", row=1, col=2)
fig.update_layout(height=430, margin=dict(t=70))
mostrar(fig)

**O que notar:** o ponto 6 (verde) e os quatro vizinhos (rosa) formam a linha 6 da matriz: um $4$ na diagonal e $-1$ nas colunas dos vizinhos. Os vizinhos de cima e de baixo ficam a $N$ colunas de distância, por isso a matriz tem cinco "faixas". Das $16 \times 16 = 256$ entradas, só 64 não são zero. Para $N = 100$, seriam $5 \times 10^4$ entradas não nulas numa matriz de $10^8$ entradas: 99,95 % de zeros.

## 3. Formulação matemática

### 3.1 Eliminação de Gauss é uma fatoração

Um sistema $A\mathbf{x} = \mathbf{b}$ com $A$ **triangular superior** se resolve de baixo para cima, uma incógnita por vez. É a **substituição regressiva**. A eliminação de Gauss transforma o sistema geral num triangular: em cada coluna $k$, subtrai da linha $i$ a linha $k$ multiplicada pelo fator

$$
\ell_{ik} = \frac{a_{ik}}{a_{kk}},
$$

o que zera tudo abaixo da diagonal. O número $a_{kk}$ é o **pivô**.

O ponto central é que os fatores $\ell_{ik}$ não precisam ser jogados fora. Guardados numa matriz triangular inferior $L$, com 1 na diagonal, eles reconstroem a matriz original:

$$
A = L\,U .
$$ (eq-lu)

Resolver $A\mathbf{x} = \mathbf{b}$ vira resolver dois sistemas triangulares: $L\mathbf{y} = \mathbf{b}$ (substituição progressiva) e depois $U\mathbf{x} = \mathbf{y}$ (regressiva). A vantagem aparece quando há vários lados direitos com a mesma matriz, por exemplo várias condições de borda na mesma placa: a fatoração, que é a parte cara, é feita uma vez só.

### 3.2 Quanto custa

Para uma matriz cheia $n \times n$:

| Etapa | Operações (aproximadamente) |
| --- | --- |
| fatoração $A = LU$ | $\tfrac{2}{3}\,n^3$ |
| cada substituição | $n^2$ |
| calcular $A^{-1}$ | $2\,n^3$ |

O $n^3$ domina tudo. Dobrar o tamanho do sistema multiplica o tempo da fatoração por 8. Para a placa com $N = 100$, ou $n = 10^4$, uma fatoração cheia faria cerca de $7 \times 10^{11}$ operações e guardaria $10^8$ números, 800 MB, quase todos zeros. É por isso que sistemas esparsos pedem outros métodos (seções 3.5 a 3.7).

Calcular a inversa para depois fazer $A^{-1}\mathbf{b}$ custa três vezes mais que a fatoração e é menos preciso. **Quase nunca se calcula a inversa** para resolver um sistema. Em NumPy, use `np.linalg.solve(A, b)`, não `np.linalg.inv(A) @ b`.

### 3.3 Pivotamento

Se um pivô $a_{kk}$ for zero, a eliminação divide por zero. Se for apenas **pequeno**, o fator $\ell_{ik}$ fica enorme e os erros de arredondamento são amplificados. O remédio é o **pivotamento parcial**: antes de eliminar a coluna $k$, trocar a linha $k$ pela linha que tem o maior $|a_{ik}|$ nessa coluna. Assim todo fator fica com $|\ell_{ik}| \le 1$.

As trocas de linha ficam registradas numa matriz de permutação $P$, e a fatoração passa a ser

$$
P A = L U .
$$

É isso que `scipy.linalg.lu` e `np.linalg.solve` fazem por dentro. A seção 7.1 mostra o que acontece sem pivotamento.

### 3.4 Condicionamento: quanto o problema amplifica erros

Os dados de um problema real nunca são exatos: $\mathbf{b}$ vem de medidas, e $A$ de propriedades de materiais. Se $\mathbf{b}$ muda um pouco, quanto muda $\mathbf{x}$? A resposta é o **número de condição**

$$
\kappa(A) = \|A\|\,\|A^{-1}\| \ \ge 1,
\qquad
\frac{\|\delta \mathbf{x}\|}{\|\mathbf{x}\|} \le \kappa(A)\, \frac{\|\delta \mathbf{b}\|}{\|\mathbf{b}\|}.
$$ (eq-condicao)

Na prática, vale uma regra simples: **resolver um sistema em ponto flutuante perde cerca de $\log_{10}\kappa(A)$ algarismos significativos**. Com $\varepsilon_{\text{mach}} \approx 10^{-16}$ e $\kappa = 10^{10}$, sobram uns 6 algarismos corretos.

O condicionamento é uma propriedade do **problema**, não do algoritmo, como vimos no capítulo 2. Nenhum algoritmo recupera os algarismos que o problema perde.

Há uma consequência importante para a verificação. O **resíduo** $\mathbf{r} = \mathbf{b} - A\hat{\mathbf{x}}$ é fácil de calcular, mas o que interessa é o **erro** $\hat{\mathbf{x}} - \mathbf{x}$. Os dois estão ligados por

$$
\frac{\|\hat{\mathbf{x}} - \mathbf{x}\|}{\|\mathbf{x}\|} \le \kappa(A)\,\frac{\|\mathbf{r}\|}{\|\mathbf{b}\|}.
$$ (eq-residuo-erro)

Num sistema mal condicionado, **um resíduo pequeno não garante um erro pequeno**.

### 3.5 Métodos iterativos clássicos

Para sistemas grandes e esparsos, em vez de fatorar, podemos **melhorar um chute** repetidamente. A equação {eq}`eq-placa` sugere o método mais simples: trocar cada $T_{i,j}$ pela média dos vizinhos e repetir.

- **Jacobi**: calcula todos os valores novos a partir dos antigos.
- **Gauss-Seidel**: usa cada valor novo assim que ele fica pronto. Costuma convergir duas vezes mais rápido que Jacobi.

Escrevendo $A = D + L + U$ (diagonal, triangular inferior e superior estritas), Gauss-Seidel é

$$
(D + L)\,\mathbf{x}^{(k+1)} = \mathbf{b} - U\,\mathbf{x}^{(k)} .
$$

O erro $\mathbf{e}^{(k)} = \mathbf{x}^{(k)} - \mathbf{x}$ é multiplicado a cada passo pela matriz $M = -(D+L)^{-1}U$, e o método converge se e só se o **raio espectral** $\rho(M)$, o maior $|\lambda|$ dos autovalores de $M$, for menor que 1. Uma condição suficiente, fácil de conferir, é a matriz ser **diagonal dominante**: $|a_{ii}| > \sum_{j \ne i} |a_{ij}|$ em toda linha.

O problema é a velocidade. Para a placa, o raio espectral de Gauss-Seidel é

$$
\rho = \cos^2\!\left(\frac{\pi}{N+1}\right) \approx 1 - \frac{\pi^2}{(N+1)^2},
$$ (eq-raio-gs)

muito perto de 1. O erro cai por um fator $\rho$ a cada iteração, então reduzir o erro por um fator fixo exige um número de iterações proporcional a $N^2$. **Refinar a grade deixa o método mais lento**, porque a informação da borda anda só um ponto por iteração.

### 3.6 Quando parar: variação pequena não é erro pequeno

O critério de parada mais comum, e o mais perigoso, é parar quando a variação entre duas iterações, $\Delta_k = \|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\|$, fica pequena. Se o erro cai por um fator $\rho$ a cada passo, então $\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)} = \mathbf{e}^{(k+1)} - \mathbf{e}^{(k)}$, e a soma dos passos que ainda faltam dá

$$
\|\mathbf{e}^{(k+1)}\| \approx \frac{\rho}{1-\rho}\,\Delta_k .
$$ (eq-erro-variacao)

Com $\rho = 0{,}999$, o erro é **mil vezes** maior que a última variação. Um critério seguro usa o **resíduo relativo** $\|\mathbf{b} - A\mathbf{x}^{(k)}\| / \|\mathbf{b}\|$, que mede quanto falta para resolver as equações, combinado com {eq}`eq-residuo-erro`.

### 3.7 Gradiente conjugado

Se $A$ é **simétrica positiva definida**, como na placa, resolver $A\mathbf{x} = \mathbf{b}$ equivale a **minimizar** a função quadrática

$$
\phi(\mathbf{x}) = \tfrac12\,\mathbf{x}^{\mathsf T} A\,\mathbf{x} - \mathbf{b}^{\mathsf T}\mathbf{x},
$$

cujo gradiente é $A\mathbf{x} - \mathbf{b} = -\mathbf{r}$. O **gradiente conjugado** (CG) desce por direções escolhidas de modo que cada passo não desfaz o progresso dos anteriores. Em aritmética exata, termina em no máximo $n$ passos. Na prática, é usado como método iterativo, e o número de iterações cresce com $\sqrt{\kappa(A)}$. Para a placa, $\kappa \sim N^2$, então o CG precisa de um número de iterações proporcional a $N$, contra $N^2$ de Gauss-Seidel. Cada iteração custa basicamente um produto matriz-vetor, barato numa matriz esparsa. O capítulo 5 volta à ideia de descer pelo gradiente.

## 4. À mão

Resolva por eliminação de Gauss com pivotamento parcial:

$$
\begin{aligned}
x_1 + 2x_2 + x_3 &= 8\\
3x_1 + x_2 - x_3 &= 2\\
2x_1 - x_2 + 3x_3 &= 9
\end{aligned}
$$

**Coluna 1.** O maior $|a_{i1}|$ é 3, na linha 2, então trocamos as linhas 1 e 2. Os fatores são $\ell_{21} = 1/3$ e $\ell_{31} = 2/3$:

$$
\left[\begin{array}{ccc|c} 3 & 1 & -1 & 2\\ 1 & 2 & 1 & 8\\ 2 & -1 & 3 & 9 \end{array}\right]
\longrightarrow
\left[\begin{array}{ccc|c} 3 & 1 & -1 & 2\\ 0 & 5/3 & 4/3 & 22/3\\ 0 & -5/3 & 11/3 & 23/3 \end{array}\right].
$$

**Coluna 2.** Os candidatos têm o mesmo módulo, $|5/3| = |-5/3|$, e mantemos a ordem. O fator é $\ell_{32} = -1$:

$$
\left[\begin{array}{ccc|c} 3 & 1 & -1 & 2\\ 0 & 5/3 & 4/3 & 22/3\\ 0 & 0 & 5 & 15 \end{array}\right].
$$

**Substituição regressiva.** $x_3 = 15/5 = 3$; $x_2 = (22/3 - 4/3 \cdot 3)/(5/3) = 2$; $x_1 = (2 - 2 + 3)/3 = 1$.

Os fatores formam $L$, e a troca de linhas forma $P$:

$$
P = \begin{bmatrix} 0 & 1 & 0\\ 1 & 0 & 0\\ 0 & 0 & 1\end{bmatrix},\quad
L = \begin{bmatrix} 1 & 0 & 0\\ 1/3 & 1 & 0\\ 2/3 & -1 & 1\end{bmatrix},\quad
U = \begin{bmatrix} 3 & 1 & -1\\ 0 & 5/3 & 4/3\\ 0 & 0 & 5\end{bmatrix}.
$$

O computador confirma:

In [ ]:
A = np.array([[1.0, 2, 1], [3, 1, -1], [2, -1, 3]])
b = np.array([8.0, 2, 9])
P, L, U = lu(A)
print("L =\n", L.round(4))
print("U =\n", U.round(4))
print("P A = L U?", np.allclose(P @ A, L @ U))
print("x =", resolve_lu(P, L, U, b))

## 5. Explore

### 5.1 O calor entrando na placa

Gauss-Seidel começa com a placa toda a $0\ ^\circ$C e, a cada iteração, troca cada ponto pela média dos vizinhos. Arraste o slider para ver o calor da borda de cima "descendo" pela grade. Repare quantas iterações são necessárias para o centro chegar perto de $25\ ^\circ$C.

In [ ]:
def gauss_seidel_placa(N, iteracoes):
    # Gauss-Seidel na placa, guardando alguns instantes para o slider
    T = np.zeros((N + 2, N + 2))
    T[-1, 1:-1] = 100.0
    fotos = {0: T.copy()}
    for k in range(1, max(iteracoes) + 1):
        for i in range(1, N + 1):
            for j in range(1, N + 1):
                T[i, j] = 0.25 * (T[i - 1, j] + T[i + 1, j] + T[i, j - 1] + T[i, j + 1])
        if k in iteracoes:
            fotos[k] = T.copy()
    return fotos

N_explore = 31
marcos = [0, 1, 5, 20, 50, 100, 200, 500, 1000]
fotos = gauss_seidel_placa(N_explore, marcos)
xs = np.linspace(0, 1, N_explore + 2)

def quadro(k):
    T = fotos[k]
    fig = go.Figure(go.Heatmap(z=T, x=xs, y=xs, zmin=0, zmax=100, colorscale="Inferno",
                               colorbar=dict(title="T (°C)")))
    c = N_explore // 2 + 1
    fig.add_trace(go.Scatter(x=[xs[c]], y=[xs[c]], mode="markers", marker=dict(color=CORES["destaque"], size=10),
                             name="centro", showlegend=False))
    fig.update_xaxes(title="x (m)", range=[0, 1])
    fig.update_yaxes(title="y (m)", range=[0, 1], scaleanchor="x")
    fig.update_layout(title=f"Gauss-Seidel, {k} iterações: centro a {numero(T[c, c], 2)} °C", width=560, height=520)
    return fig

quadros = [(str(k), quadro(k)) for k in marcos]
mostrar(com_controle(quadros, rotulo="iterações = ", ativo=5, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** depois de 100 iterações, o centro ainda está longe dos $25\ ^\circ$C que ele deve ter (seção 6.2). O calor se espalha devagar, um ponto da grade por iteração. Com uma grade duas vezes mais fina, seriam necessárias cerca de quatro vezes mais iterações.

### 5.2 Condicionamento visto de perto

Um sistema $2 \times 2$ é a interseção de duas retas. Se o lado direito tem uma incerteza, cada reta vira uma **faixa**, e a solução pode estar em qualquer ponto da interseção das faixas. Arraste o slider para diminuir o ângulo entre as retas.

In [ ]:
def figura_condicao(angulo_graus, delta=0.05):
    th = np.radians(angulo_graus)
    # reta 1: y = 0;  reta 2: passa pela origem com inclinação th.  Normais unitárias:
    normais = [np.array([0.0, 1.0]), np.array([-np.sin(th), np.cos(th)])]
    A = np.vstack(normais)
    kappa = np.linalg.cond(A)
    t = np.linspace(-3, 3, 2)
    fig = go.Figure()
    for nrm, cor, nome in zip(normais, [CORES["aproximacao"], CORES["secundaria"]], ["equação 1", "equação 2"]):
        d = np.array([nrm[1], -nrm[0]])
        for s, mostra in [(-delta, False), (delta, False), (0.0, True)]:
            p = s * nrm + np.outer(t, d)
            fig.add_trace(go.Scatter(x=p[:, 0], y=p[:, 1], mode="lines", name=nome, showlegend=mostra,
                                     line=dict(color=cor, width=2 if s == 0 else 1, dash="solid" if s == 0 else "dot")))
    cantos = np.array([np.linalg.solve(A, [s1, s2]) for s1, s2 in [(-delta, -delta), (-delta, delta), (delta, delta), (delta, -delta), (-delta, -delta)]])
    fig.add_trace(go.Scatter(x=cantos[:, 0], y=cantos[:, 1], fill="toself", mode="lines", name="soluções possíveis",
                             line=dict(color=CORES["erro"], width=1), fillcolor="rgba(213,94,0,0.35)"))
    fig.update_xaxes(title="x₁", range=[-1.5, 1.5])
    fig.update_yaxes(title="x₂", range=[-0.75, 0.75], scaleanchor="x")
    fig.update_layout(title=f"ângulo {angulo_graus}°: κ = {numero(kappa, 1)}, incerteza na solução ≈ {numero(np.abs(cantos[:, 0]).max() / delta, 1)} × a dos dados",
                      height=420)
    return fig

angulos = [90, 60, 45, 30, 20, 10, 5, 2]
quadros = [(f"{a}°", figura_condicao(a)) for a in angulos]
mostrar(com_controle(quadros, rotulo="ângulo entre as retas = ", ativo=5, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com retas perpendiculares, $\kappa = 1$ e a incerteza da solução é a dos dados. Com retas quase paralelas, a região de soluções possíveis se estica ao longo das retas: um sistema **mal condicionado** é um sistema cujas equações são quase redundantes. Nenhum método numérico encolhe essa região.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Escreva em Python a função `temperatura_placa(N, tol)` que calcula a temperatura nos $N \times N$ pontos internos de uma placa quadrada com a borda de cima a 100 °C e as outras a 0 °C, pelo método de Gauss-Seidel. Preciso da temperatura com precisão de `tol` = 0,001 °C. Devolva a matriz de temperaturas, incluindo as bordas, e o número de iterações."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
def temperatura_placa(N, tol=1e-3, max_iter=100_000):
    T = np.zeros((N + 2, N + 2))
    T[-1, 1:-1] = 100.0                      # borda de cima
    for it in range(max_iter):
        maior_variacao = 0.0
        for i in range(1, N + 1):
            for j in range(1, N + 1):
                novo = 0.25 * (T[i - 1, j] + T[i + 1, j] + T[i, j - 1] + T[i, j + 1])
                maior_variacao = max(maior_variacao, abs(novo - T[i, j]))
                T[i, j] = novo
        if maior_variacao < tol:             # convergiu com a precisão pedida
            break
    return T, it + 1

O código está limpo, e o método é o pedido. O comentário diz "convergiu com a precisão pedida". Será?

### 6.2 Verificação: uma solução conhecida

Não há fórmula simples para $T(x, y)$, mas há um valor exato fácil de achar. Gire a placa de $90^\circ$ quatro vezes, cada vez com uma borda diferente a $100\ ^\circ$C. Somando as quatro soluções, todas as bordas ficam a $100\ ^\circ$C, e a soma é $100\ ^\circ$C em toda a placa. Por simetria, as quatro soluções têm o **mesmo valor no centro**, que portanto é exatamente $100/4 = 25\ ^\circ$C. O argumento vale também para as equações discretas, então o sistema linear, resolvido exatamente, dá $25\ ^\circ$C no ponto central para todo $N$ ímpar.

In [ ]:
print(" N   iterações   centro (°C)    erro no centro (°C)")
resultados = {}
for N in [7, 15, 31, 63]:
    T, it = temperatura_placa(N)
    c = N // 2 + 1
    resultados[N] = (T, it)
    print(f"{N:2d}   {it:9d}   {T[c, c]:11.4f}    {abs(T[c, c] - 25):.4f}")

**O que notar:** o erro no centro é muito maior que os $0{,}001\ ^\circ$C pedidos, e **cresce** quando a grade é refinada. Com $N = 63$, o erro passa de $0{,}4\ ^\circ$C, mais de 400 vezes a tolerância. O código "convergiu", mas para um número errado.

### 6.3 Por que: a relação entre variação e erro

O agente usou a maior variação entre duas iterações como se fosse o erro. A equação {eq}`eq-erro-variacao` mostra que o erro é cerca de $\rho/(1-\rho)$ vezes essa variação, e {eq}`eq-raio-gs` dá $1/(1-\rho) \approx (N+1)^2/\pi^2$. Vamos comparar essa previsão com o erro medido em todos os pontos da placa, usando como referência a solução do sistema esparso por um método direto:

In [ ]:
print(" N   erro máximo medido   previsão tol·ρ/(1−ρ)")
erros_medidos, previsoes = [], []
for N, (T, it) in resultados.items():
    A, b = placa_laplace(N)
    T_ref = spsolve(A.tocsc(), b).reshape(N, N)
    erro = np.abs(T[1:-1, 1:-1] - T_ref).max()
    rho = np.cos(np.pi / (N + 1)) ** 2
    erros_medidos.append(erro)
    previsoes.append(1e-3 * rho / (1 - rho))
    print(f"{N:2d}   {erro:18.4f}   {previsoes[-1]:20.4f}")

**O que notar:** a previsão acerta a ordem de grandeza em todos os casos, e acerta também a tendência: o erro cresce como $N^2$. O critério de parada do agente, e não o método, é o que está errado. O teste que o próprio agente costuma escrever, com uma grade pequena como $N = 7$, passa sem problema: o erro ali é de $0{,}005\ ^\circ$C.

### 6.4 A correção

Há duas correções, e é bom conhecer as duas.

1. **Parar pelo resíduo relativo**, $\|\mathbf{b} - A\mathbf{x}\|/\|\mathbf{b}\| < \text{tol}$, ou pelo menos dividir a variação por $1-\rho$. O código continua correto, mas fica lento.
2. **Trocar de método.** Para um sistema esparso e simétrico positivo definido, o gradiente conjugado ou um método direto esparso são muito mais rápidos.

O gráfico compara o número de iterações para reduzir o resíduo relativo a $10^{-8}$, medido com as implementações de referência do pacote `cnlivro`:

In [ ]:
def iteracoes_gs_esparso(A, b, tol):
    # Gauss-Seidel em forma matricial: (D + L) x_novo = b − U x
    from scipy.sparse.linalg import spsolve_triangular
    DL = sparse.tril(A, format="csr")
    U = sparse.triu(A, k=1, format="csr")
    x = np.zeros_like(b)
    nb = np.linalg.norm(b)
    for k in range(1, 200_000):
        x = spsolve_triangular(DL, b - U @ x, lower=True)
        if np.linalg.norm(b - A @ x) / nb < tol:
            return k

Ns_gs = [7, 15, 31]
Ns_cg = [7, 15, 31, 63, 127]
it_gs = [iteracoes_gs_esparso(*placa_laplace(N), 1e-8) for N in Ns_gs]
it_cg = [gradiente_conjugado(*placa_laplace(N), tol=1e-8).iteracoes for N in Ns_cg]
for N, k in zip(Ns_gs, it_gs):
    print(f"N = {N:3d}: Gauss-Seidel {k:6d} iterações")
for N, k in zip(Ns_cg, it_cg):
    print(f"N = {N:3d}: gradiente conjugado {k:4d} iterações")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=Ns_gs, y=it_gs, mode="lines+markers", name="Gauss-Seidel", line=dict(color=CORES["erro"])))
fig.add_trace(go.Scatter(x=Ns_cg, y=it_cg, mode="lines+markers", name="gradiente conjugado", line=dict(color=CORES["aproximacao"])))
for p, dash, y0 in [(2, "dot", 2.5 * it_gs[0]), (1, "dash", 0.4 * it_cg[0])]:
    xr = np.array([7, 127])
    fig.add_trace(go.Scatter(x=xr, y=y0 * (xr / 7.0) ** p, mode="lines", name=f"∝ N{'²' if p == 2 else ''}",
                             line=dict(color=CORES["referencia"], dash=dash, width=1)))
fig.update_xaxes(type="log", title="N (a grade tem N × N pontos)", tickvals=Ns_cg)
fig.update_yaxes(type="log", title="iterações até resíduo < 10⁻⁸", exponentformat="power")
fig.update_layout(title="Gauss-Seidel precisa de ∝ N² iterações; o gradiente conjugado, de ∝ N")
mostrar(fig)

Por fim, a versão que um engenheiro usaria: montar a matriz esparsa e chamar um resolvedor pronto. Para $N = 63$, são quase 4 000 incógnitas:

In [ ]:
N = 63
A, b = placa_laplace(N)
T = spsolve(A.tocsc(), b).reshape(N, N)
residuo = np.linalg.norm(b - A @ T.ravel()) / np.linalg.norm(b)
print(f"centro: {T[N // 2, N // 2]:.10f} °C    resíduo relativo: {residuo:.1e}")

> **Lição.** "Convergiu" não quer dizer "está certo". Num método iterativo, o critério de parada é parte do método e precisa ser verificado como o resto. Peça sempre o histórico do **resíduo** e teste com uma grade **grande**, não só com a de exemplo.

## 7. Quando falha

### 7.1 Sem pivotamento

O sistema $\varepsilon x_1 + x_2 = 1$, $x_1 + x_2 = 2$, com $\varepsilon = 10^{-17}$, é perfeitamente bem condicionado ($\kappa \approx 2{,}6$), e a solução é quase exatamente $x_1 = x_2 = 1$. Sem pivotamento, o fator é $\ell_{21} = 1/\varepsilon = 10^{17}$, e o $1$ da segunda equação some diante de $10^{17}$:

In [ ]:
eps = 1e-17
A = np.array([[eps, 1.0], [1.0, 1.0]])
b = np.array([1.0, 2.0])
print("sem pivotamento:", eliminacao_gauss(A, b, pivotamento=False))
print("com pivotamento:", eliminacao_gauss(A, b, pivotamento=True))
print(f"κ(A) = {np.linalg.cond(A):.2f}")

O problema é bom; o **algoritmo** sem pivotamento é instável. O erro é de 100 %, sem nenhum aviso.

### 7.2 Mal condicionamento: a matriz de Hilbert

A matriz de Hilbert, $H_{ij} = 1/(i + j - 1)$, aparece no ajuste de polinômios por mínimos quadrados (capítulo 7) e é famosa por ser mal condicionada. Escolhemos $\mathbf{x} = (1, 1, \ldots, 1)$, calculamos $\mathbf{b} = H\mathbf{x}$ e pedimos ao computador para resolver de volta, com o melhor algoritmo disponível:

In [ ]:
from scipy.linalg import hilbert
ns = np.arange(2, 14)
kappas, erros, residuos = [], [], []
for n in ns:
    H = hilbert(n)
    x = np.ones(n)
    b = H @ x
    x_hat = np.linalg.solve(H, b)
    kappas.append(np.linalg.cond(H))
    erros.append(np.linalg.norm(x_hat - x) / np.linalg.norm(x))
    residuos.append(np.linalg.norm(b - H @ x_hat) / np.linalg.norm(b) or np.nan)   # resíduo 0 não aparece no log

In [ ]:
eps = np.finfo(float).eps
fig = go.Figure()
fig.add_trace(go.Scatter(x=kappas, y=erros, mode="lines+markers", name="erro relativo", line=dict(color=CORES["erro"]),
                         text=[f"n = {n}" for n in ns]))
fig.add_trace(go.Scatter(x=kappas, y=residuos, mode="lines+markers", name="resíduo relativo",
                         line=dict(color=CORES["aproximacao"]), text=[f"n = {n}" for n in ns]))
fig.add_trace(go.Scatter(x=kappas, y=eps * np.array(kappas), mode="lines", name="κ · ε_mach",
                         line=dict(color=CORES["referencia"], dash="dash")))
fig.update_xaxes(type="log", title="número de condição κ(H)", exponentformat="power")
fig.update_yaxes(type="log", title="relativo", exponentformat="power", range=[-18, 2])
fig.update_layout(title="Resíduo sempre minúsculo, erro do tamanho de κ · ε")
mostrar(fig)

**O que notar:** o resíduo fica abaixo de $10^{-15}$ em todos os casos: `np.linalg.solve` fez o seu trabalho perfeitamente. Mas o erro acompanha a reta $\kappa\,\varepsilon_{\text{mach}}$, como prevê {eq}`eq-residuo-erro`. Com $n = 13$, $\kappa \approx 5 \times 10^{17}$, e não sobra nenhum algarismo correto. O resíduo, sozinho, teria declarado o resultado perfeito.

A saída não é um algoritmo melhor, e sim um problema melhor: reformular o modelo, por exemplo trocando a base de polinômios $1, x, x^2, \ldots$ por polinômios ortogonais (capítulo 7). Por isso, antes de confiar numa solução, calcule `np.linalg.cond(A)`.

### 7.3 Jacobi e Gauss-Seidel podem divergir

Se a matriz não é diagonal dominante, nada garante a convergência:

In [ ]:
A = np.array([[1.0, 3.0], [2.0, 1.0]])
b = np.array([4.0, 3.0])
r = gauss_seidel(A, b, tol=1e-10, max_iter=30)
print(r)
print("resíduos relativos:", [f"{e:.1e}" for e in r.erros[:6]], "...")
print("solução de verdade:", np.linalg.solve(A, b))

Trocar a ordem das equações, aqui, torna a matriz diagonal dominante e resolve o problema. Num sistema grande, isso raramente é possível, e o gradiente conjugado (para matrizes simétricas positivas definidas) ou o GMRES (para as demais) são as escolhas usuais.

### 7.4 Matrizes cheias grandes demais

Para a placa com $N = 300$, são $n = 9 \times 10^4$ incógnitas. A matriz cheia teria $8{,}1 \times 10^9$ entradas, ou 65 GB, e a fatoração faria cerca de $5 \times 10^{14}$ operações. A matriz esparsa tem $4{,}5 \times 10^5$ entradas não nulas, uns 5 MB. Trate uma matriz esparsa com ferramentas esparsas (`scipy.sparse`) desde a montagem: um `A.toarray()` esquecido no meio do código é suficiente para esgotar a memória.

## Resumo

| Método | Para quê | Custo | Cuidado |
| --- | --- | --- | --- |
| Eliminação de Gauss / $PA = LU$ | matrizes cheias, até alguns milhares | $\tfrac23 n^3$ uma vez, $n^2$ por lado direito | sempre com pivotamento |
| Inversa $A^{-1}$ | quase nunca | $2n^3$ | mais cara e menos precisa que LU |
| Jacobi, Gauss-Seidel | sistemas esparsos simples; suavizadores | iterações $\propto 1/(1-\rho)$, $\propto N^2$ na placa | parar pelo resíduo, não pela variação |
| Gradiente conjugado | esparsas simétricas positivas definidas | iterações $\propto \sqrt{\kappa}$, $\propto N$ na placa | só para $A$ SPD |
| Direto esparso (`spsolve`) | esparsas de tamanho moderado | depende do preenchimento | memória em 3D |

| Conceito | Ideia central |
| --- | --- |
| Número de condição | $\kappa(A) = \|A\|\|A^{-1}\|$; perde-se $\approx \log_{10}\kappa$ algarismos |
| Resíduo × erro | erro relativo $\le \kappa \times$ resíduo relativo |
| Raio espectral | o erro de um método iterativo cai por $\rho$ a cada passo |

## Exercícios

**Conceituais (sem computador)**

1. Mostre que, ao fazer a eliminação de Gauss, os fatores $\ell_{ik}$ formam uma matriz $L$ com $LU = A$, para uma matriz $3 \times 3$ sem trocas de linha.
2. Conte as operações da eliminação de Gauss e confirme o termo dominante $\tfrac23 n^3$.
3. Calcule $\kappa_\infty(A)$ para $A = \begin{bmatrix} 1 & 1 \\ 1 & 1 + \delta\end{bmatrix}$ e interprete o resultado quando $\delta \to 0$ com o desenho da seção 5.2.
4. Deduza {eq}`eq-erro-variacao` supondo que o erro é multiplicado exatamente por $\rho$ a cada iteração.

**Verificação (com agente)**

5. Peça a um agente a decomposição de Cholesky, $A = R^{\mathsf T}R$, para matrizes simétricas positivas definidas. Verifique com matrizes aleatórias da forma $M^{\mathsf T}M + nI$, compare com `np.linalg.cholesky` e meça como o tempo cresce com $n$ (a inclinação no gráfico log-log deve ser 3).
6. Peça um resolvedor por SOR (sobrerrelaxação sucessiva) para a placa. Ache experimentalmente o melhor fator de relaxação $\omega$ para $N = 31$ e compare com o valor teórico $\omega^* = 2/(1 + \sin(\pi/(N+1)))$.
7. Peça a um agente para resolver a placa com a borda de cima a $100\ ^\circ$C e a da direita **isolada** (derivada nula). Que solução conhecida, ou que invariante, você usaria para verificar o código?

**Projeto aberto**

8. **Treliça de uma ponte.** Monte o sistema de equilíbrio dos nós de uma treliça plana com pelo menos 10 barras, resolva-o e calcule o número de condição. O que acontece com $\kappa$ quando uma barra fica quase alinhada com outra? Relacione com a seção 5.2.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulos 9 a 12.
- L. N. Trefethen e D. Bau, *Numerical Linear Algebra*, SIAM, 1997, aulas 20 a 23 (eliminação de Gauss e pivotamento) e 38 (gradiente conjugado).
- J. R. Shewchuk, *An Introduction to the Conjugate Gradient Method Without the Agonizing Pain*, 1994, disponível on-line.